# 環境設置

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install -q --upgrade transformers peft trl accelerate bitsandbytes scipy wandb

# Import Packages

In [ ]:
import torch, os
from peft import LoraConfig
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
import os
os.environ["UNSLOTH_DISABLE_FUSED_CROSS_ENTROPY"] = "1"

from unsloth import FastLanguageModel
import torch


Unsloth 微調參數設置

In [ ]:
max_seq_length = 2048
dtype = None # 自動偵測使用 `None`/ Tesla T4 & V100 用 Float16 / Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

Unsloth 模型設置 (Qwen3-4B)

In [ ]:

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    attn_implementation = "eager",
)

/tmp/ipython-input-2002776035.py:4: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.1: Fast Qwen3 patching. Transformers: 4.57.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 的 LoRA 設置

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # 官方說 0 is optimized
    bias = "none",    #  官方說 "none" 是 optimized
    use_gradient_checkpointing = False,
    random_state = 3407,
    use_rslora = False,  # 本次不使用 rank stabilized LoRA
    loftq_config = None, # 本次不使用 LoftQ
)

Unsloth 2025.10.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# 訓練資料預處理

In [ ]:
# 假設沒有下載作業的數據的話可以 uncomment 來使用
# !gdown --id 1yFaK4fJRfCeyWBQCehqMFd2S3UzDHywr
# !unzip hw2.zip

設定function

In [ ]:
EOS_TOKEN = tokenizer.eos_token  # 必須加上 EOS_TOKEN
SYS_PROMPT = "你是一個中文語言專家，你將進行文言文與白話文之間的精準翻譯。翻譯時，請產生最符合歷史語言習慣的中文輸出；請注意，只能使用繁體中文；請在回答時保持簡潔，不需要任何其他回饋。以下是你的任務"

def preprocess(examples, tokenizer):
    """
    手動創建 input_ids 和 labels
    - 問題部分的 labels = -100 (不計算 loss)
    - 答案部分的 labels = token_ids (計算 loss)
    """
    instructions = examples["instruction"]
    outputs = examples["output"]

    all_input_ids = []
    all_labels = []
    all_attention_mask = []

    for instruction, output in zip(instructions, outputs):

        # 構建問題和答案部分
        question_text = SYS_PROMPT+instruction
        answer_text = output + EOS_TOKEN

        # 分別 tokenize 問題和答案
        question_tokens = tokenizer(
            question_text,
            add_special_tokens=True,  # 添加 BOS token
            truncation=False,
            padding=False,
        )
        answer_tokens = tokenizer(
            answer_text,
            add_special_tokens=False,  # 不添加特殊 token
            truncation=False,
            padding=False,
        )

        # question 和 answer input_ids 加起來
        input_ids = question_tokens["input_ids"] + answer_tokens["input_ids"]

        # 設定 labels: 問題部分=-100, 答案部分=實際token ids
        labels = (
            [-100] * len(question_tokens["input_ids"]) +  # 問題部分不計算loss
            answer_tokens["input_ids"]            # 答案部分計算loss
        )

        # 設定 attention_mask
        attention_mask = [1] * len(input_ids)

        # 7. 檢查長度並截斷(如果需要)
        if len(input_ids) > max_seq_length:
            input_ids = input_ids[:max_seq_length]
            labels = labels[:max_seq_length]
            attention_mask = attention_mask[:max_seq_length]

        all_input_ids.append(input_ids)
        all_labels.append(labels)
        all_attention_mask.append(attention_mask)

    return {
        "input_ids": all_input_ids,
        "labels": all_labels,
        "attention_mask": all_attention_mask,
    }

載入數據 & 執行預處理

In [ ]:
from datasets import Dataset
import pandas as pd

print("\n📊 Loading training data")
train_data_df = pd.read_json("hw3/data/train.json", lines=False)
train_ds = Dataset.from_pandas(train_data_df)
train_dataset = train_ds.map(
    lambda x: preprocess(x, tokenizer),
    batched=True,
    remove_columns=train_ds.column_names,
    num_proc=4,
    desc="Processing training data"
)


print("📊 Loading evaluation data")
eval_data_df = pd.read_json("hw3/data/public_test.json", lines=False)
eval_ds = Dataset.from_pandas(eval_data_df)
eval_dataset = eval_ds.map(
    lambda x: preprocess(x, tokenizer),
    batched=True,
    remove_columns=eval_ds.column_names,
    num_proc=4,
    desc="Processing evaluation data"
)


📊 Loading training data


Processing training data (num_proc=4):   0%|          | 0/10000 [00:00<?, ? examples/s]

📊 Loading evaluation data


Processing evaluation data (num_proc=4):   0%|          | 0/250 [00:00<?, ? examples/s]

驗證數據處理結果

In [ ]:

print("\n" + "=" * 70)
print("驗證第一個樣例:")
print("=" * 70)

sample = train_dataset[0]
input_ids = sample["input_ids"]
labels = sample["labels"]
attention_mask = sample["attention_mask"]

print(f"\n長度統計:")
print(f"  Input IDs 長度: {len(input_ids)}")
print(f"  Labels 長度: {len(labels)}")
print(f"  Attention Mask 長度: {len(attention_mask)}")

# 統計 labels
labels_tensor = torch.tensor(labels)
ignored_count = (labels_tensor == -100).sum().item()
loss_count = (labels_tensor != -100).sum().item()

print(f"\nLabels 統計:")
print(f"  被忽略的 tokens (=-100): {ignored_count}")
print(f"  計算 loss 的 tokens: {loss_count}")
print(f"  比例: {loss_count}/{len(labels)} = {loss_count/len(labels)*100:.1f}%")

# 顯示完整輸入
print(f"\n完整輸入文本:")
full_text = tokenizer.decode(input_ids, skip_special_tokens=False)
print(full_text)

# 只顯示答案部分(會計算 loss 的部分)
answer_token_ids = [tid for tid, label in zip(input_ids, labels) if label != -100]
print(f"\n答案部分(計算 loss 的部分):")
answer_text = tokenizer.decode(answer_token_ids, skip_special_tokens=True)
print(answer_text)

print("=" * 70)

# 檢查是否有問題
if loss_count == 0:
    print("\n⚠️ 錯誤: 沒有任何 token 會計算 loss!")
    print("請檢查數據格式和處理邏輯。")
    exit(1)
else:
    print(f"\n✅ 數據處理成功! 每個樣例平均有 {loss_count} 個 tokens 會計算 loss。")



驗證第一個樣例:

長度統計:
  Input IDs 長度: 137
  Labels 長度: 137
  Attention Mask 長度: 137

Labels 統計:
  被忽略的 tokens (=-100): 111
  計算 loss 的 tokens: 26
  比例: 26/137 = 19.0%

完整輸入文本:
你是一個中文語言專家，你將進行文言文與白話文之間的精準翻譯。翻譯時，請產生最符合歷史語言習慣的中文輸出；請注意，只能使用繁體中文；請在回答時保持簡潔，不需要任何其他回饋。以下是你的任務翻譯成文言文：
雅裏惱怒地說： 從前在福山田獵時，你誣陷獵官，現在又說這種話。
答案：雅裏怒曰： 昔畋於福山，卿誣獵官，今復有此言。<|im_end|>

答案部分(計算 loss 的部分):
雅裏怒曰： 昔畋於福山，卿誣獵官，今復有此言。

✅ 數據處理成功! 每個樣例平均有 26 個 tokens 會計算 loss。


Optional: 檢查會不會太長

In [ ]:
# 獲取所有樣本的長度
lengths = [len(sample['input_ids']) for sample in train_dataset]

# 定義一個安全的上限 (例如 max_seq_length 的 98%)
SAFE_MAX_LENGTH = 1999 # 如果 max_seq_length 是 2048

# 過濾掉長度超過安全上限的樣本
train_dataset = train_dataset.filter(
    lambda example: len(example['input_ids']) <= SAFE_MAX_LENGTH
)

print(len(train_dataset))

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

10000


設置 DataCollator
(`DataCollatorForSeq2Seq` 特別是對序列-序列的任務)

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,  # padding 的 labels 設為 -100
    padding=True,
)


使用 Huggingface [TRL SFT](https://huggingface.co/docs/trl/sft_trainer) 的`SFTTrainer`

In [ ]:
from trl import SFTTrainer,SFTConfig
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch
from transformers import Trainer

# 配置訓練參數
training_args = TrainingArguments(
    # 一些超參數
    per_device_train_batch_size = 2,
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps = 4,

    warmup_steps = 5,
    warmup_ratio = 0.03,
    max_steps = 2000,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    gradient_checkpointing = False,

    optim = "adamw_torch", # QLoRA 適用
    weight_decay = 0.01,
    lr_scheduler_type = "cosine", # 翻譯任務推薦使用 cosine
    seed = 3407,
    # 評估配置
    eval_strategy = "steps", # 或 "epoch"
    eval_steps = 50, # 每 50 步評估一次
    save_strategy = "steps",
    save_steps = 50,
    save_total_limit = 3, # 只保留最好的 3 個 checkpoint
    load_best_model_at_end = True, # 訓練結束時載入最佳模型
    metric_for_best_model = "eval_loss", # 根據 eval_loss 選擇最佳模型
    greater_is_better = False,

    output_dir = "ntu_hw2_qwen3_3",
    run_name = "qwen-classical-translation",
    logging_steps = 10,
    logging_first_step = True,
    report_to = "wandb",
)

# 創建 Trainer (使用 data_collator)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    data_collator = data_collator,  # 使用 data_collator
    args = training_args,
    max_seq_length = max_seq_length,
    packing = False,  # 關閉 packing,使用手動標記的數據
)


使用 wandb 來記錄訓練過程

In [ ]:
import wandb
wandb.finish()
wandb.init()

wandb: Currently logged in as: soaring0616 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [ ]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 2 | Total steps = 2,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss,Validation Loss
50,1.706900,1.660495
100,1.743800,1.627725
150,1.846700,1.603221
200,1.654300,1.603117
250,1.616900,1.574263
300,1.498900,1.560238
350,1.658900,1.554769
400,1.678000,1.532573
450,1.928700,1.524454
500,1.498600,1.527840


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=2000, training_loss=1.3575092499256134, metrics={'train_runtime': 7354.748, 'train_samples_per_second': 2.175, 'train_steps_per_second': 0.272, 'total_flos': 5.28265617437184e+16, 'train_loss': 1.3575092499256134, 'epoch': 1.6})

# 存檔

In [ ]:
!zip -r ntu_hw2_qwen3_3.zip /content/ntu_hw2_qwen3_3

  adding: content/ntu_hw2_qwen3_3/ (stored 0%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/ (stored 0%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/trainer_state.json (deflated 80%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/tokenizer.json (deflated 81%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/added_tokens.json (deflated 68%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/training_args.bin (deflated 53%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/adapter_config.json (deflated 57%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/special_tokens_map.json (deflated 69%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/scheduler.pt (deflated 61%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/vocab.json (deflated 61%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/scaler.pt (deflated 64%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/tokenizer_config.json (deflated 90%)
  adding: content/ntu_hw2_qwen3_3/checkpoint-1250/chat_template.jinja 

In [ ]:
from huggingface_hub import upload_file

repo_id = "soaring0616/ntu_hw2_qwen3_3"  # 換成你的repo
path_in_repo = "ntu_hw2_qwen3_3.zip"   # 存在HF上要叫什麼名字
path_local = "/content/ntu_hw2_qwen3_3.zip"  # Colab中的路徑

upload_file(
    path_or_fileobj=path_local,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model"  # 或 "dataset" 或 "space"
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/ntu_hw2_qwen3_3.zip:   0%|          |  524kB / 2.19GB            

CommitInfo(commit_url='https://huggingface.co/soaring0616/ntu_hw2_qwen3_3/commit/1eba74a1928e6222566d5024fd58a346fce2d6b3', commit_message='Upload ntu_hw2_qwen3_3.zip with huggingface_hub', commit_description='', oid='1eba74a1928e6222566d5024fd58a346fce2d6b3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/soaring0616/ntu_hw2_qwen3_3', endpoint='https://huggingface.co', repo_type='model', repo_id='soaring0616/ntu_hw2_qwen3_3'), pr_revision=None, pr_num=None)

In [ ]:
# Alternatively
# --------------------
# model.save_pretrained("lora_model") # Local saving
# tokenizer.save_pretrained("lora_model")
# model.push_to_hub("soaring0616/hw2-qwen_2") # Online saving
# tokenizer.push_to_hub("soaring0616/hw2-qwen_2") # Online saving

## 將 Unsloth 的 adpater 轉成 作業環境可執行的 adapter

In [ ]:
from huggingface_hub import HfApi
from peft import PeftModel, LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import json
import os

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("📥 載入原始 adapter...")
BASE_MODEL = "unsloth/Qwen3-4B"
ADAPTER_REPO = "soaring0616/ntu_hw2_qwen3_adapter"

# w/ new API
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,  # 用這個取代 load_in_4bit
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
print("✅ 原始 adapter 載入成功\n")

# Compatible Config
print("🔧 創建兼容配置...")

original_config = model.peft_config['default']

compatible_config = LoraConfig(
    r=original_config.r,
    lora_alpha=original_config.lora_alpha,
    target_modules=original_config.target_modules,
    lora_dropout=original_config.lora_dropout,
    bias=original_config.bias,
    task_type=original_config.task_type,
    inference_mode=True,
)

compatible_config.base_model_name_or_path = "Qwen/Qwen3-4B"
print("✅ 兼容配置創建完成\n")

# saving local
output_dir = "./adapter_compatible"
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
compatible_config.save_pretrained(output_dir)

print(f"✅ 兼容版本保存到: {output_dir}\n")

# check configuration
with open(f"{output_dir}/adapter_config.json", 'r') as f:
    saved_config = json.load(f)
    print("最終配置:")
    print(json.dumps(saved_config, indent=2))

# 4. 上傳到 Hugging Face
print("\n📤 上傳到 Hugging Face...")

NEW_REPO_ID = "soaring0616/ntu_hw2_qwen3_adapter_compatible"

api = HfApi()

try:
    api.create_repo(repo_id=NEW_REPO_ID, repo_type="model", exist_ok=True)
    print(f"✅ Repo 已創建: {NEW_REPO_ID}")
except Exception as e:
    print(f"✅ Repo 已存在: {NEW_REPO_ID}")

api.upload_folder(
    folder_path=output_dir,
    repo_id=NEW_REPO_ID,
    repo_type="model",
)

print(f"\n✅ 上傳完成!")
print(f"🔗 Repo 連結: https://huggingface.co/{NEW_REPO_ID}")